<a href="https://colab.research.google.com/github/APriya-spec/COMP-ANALY-HIGH-TP-BIOMED-DATA/blob/Assignment-3/HT_ASG3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import gzip
from IPython.display import display, HTML

# Load PTT file
def load_ptt_data(file_path):
    with gzip.open(file_path, 'rt') as file:
        return pd.read_csv(file, sep='\t', skiprows=3, header=None, names=[
            "Locations", "Strand", "Length", "PID", "Gene", "Synonym", "Code", "COG", "Product"])

# Identify operons in PTT
def identify_operons_ptt(df):
    operons = []
    current = [df.iloc[0]]
    for i in range(1, len(df)):
        prev, curr = df.iloc[i-1], df.iloc[i]
        end_prev = int(prev['Locations'].split('..')[-1])
        start_curr = int(curr['Locations'].split('..')[0])
        if prev['Strand'] == curr['Strand'] and (start_curr - end_prev) <= 50:
            current.append(curr)
        else:
            if len(current) > 1:
                operons.append(current)
            current = [curr]
    if len(current) > 1:
        operons.append(current)
    return operons

# Load GFF
def load_gff_data(file_path):
    records = []
    with open(file_path, 'r') as file:
        for line in file:
            if line.startswith('#') or not line.strip(): continue
            parts = line.strip().split('\t')
            attrs = {kv.split('=')[0]: kv.split('=')[1] for kv in parts[8].split(';') if '=' in kv}
            records.append([
                parts[0], int(parts[3]), int(parts[4]), parts[6],
                attrs.get('ID', 'unknown'),
                attrs.get('locus_tag', 'unknown'),
                attrs.get('product', 'unknown')
            ])
    return pd.DataFrame(records, columns=['Contig', 'Start', 'End', 'Strand', 'ID', 'LocusTag', 'Product']).sort_values(by='Start')

# Identify operons in GFF
def identify_operons_gff(df):
    operons = []
    current = [df.iloc[0]]
    for i in range(1, len(df)):
        prev, curr = df.iloc[i-1], df.iloc[i]
        if curr['Strand'] == prev['Strand'] and (curr['Start'] - prev['End']) <= 50:
            current.append(curr)
        else:
            if len(current) > 1:
                operons.append(current)
            current = [curr]
    if len(current) > 1:
        operons.append(current)
    return operons

# Print operons from PTT in scrollable HTML
def process_ptt_files(ptt_files):
    for file in ptt_files:
        df = load_ptt_data(file)
        operons = identify_operons_ptt(df)
        html = f"<b>Found {len(operons)} operons in {file}.</b><br><pre>"
        for i, operon in enumerate(operons, 1):
            genes = [(g['Gene'] if g['Gene'] != '-' and g['Gene'].strip() else f"[{g['Product']}]") for _, g in pd.DataFrame(operon).iterrows()]
            html += f"Operon {i}: {', '.join(genes)}\n"
        html += "</pre>"
        display(HTML(f"<div style='max-height: 400px; overflow-y: scroll; border: 1px solid #ccc; padding: 10px'>{html}</div>"))

# Print operons from GFF in scrollable HTML
def process_gff_file(file):
    df = load_gff_data(file)
    operons = identify_operons_gff(df)
    html = f"<b>Found {len(operons)} operons.</b><br><pre>"
    for i, operon in enumerate(operons, 1):
        genes = ', '.join(f"{g['LocusTag']} ({g['Product']})" for _, g in pd.DataFrame(operon).iterrows())
        html += f"Operon {i}: {genes}\n"
    html += "</pre>"
    display(HTML(f"<div style='max-height: 400px; overflow-y: scroll; border: 1px solid #ccc; padding: 10px'>{html}</div>"))


In [ ]:
ptt_files = [
    "E_coli_K12_MG1655.ptt.gz",
    "B_subtilis_168.ptt.gz",
    "Halobacterium_NRC1.ptt.gz",
    "Synechocystis_PCC6803_uid159873.ptt.gz"
]

gff_file = "2088090036.gff"

process_ptt_files(ptt_files)
process_gff_file(gff_file)
